# Step 1: Ingestion Pipeline

**Goal:** Load the GitLab Handbook markdown files, chunk them intelligently, embed them, and store in ChromaDB.

**What happens here:**
1. Load all `.md` files from the dataset folder
2. Split each file into chunks (paragraph-aware, with overlap)
3. Generate vector embeddings for each chunk using `all-MiniLM-L6-v2`
4. Store everything in a local ChromaDB collection

**Why this order matters:** We need to get ingestion right before anything else. Bad chunks = bad retrieval = bad answers, no matter how good the LLM is.

In [ ]:
# Install dependencies (run once)
# !pip install chromadb

In [1]:
import os
import glob

DATASET_DIR = "datasets"  # unzip the provided dataset here

## 1.1 Load the Markdown Files

Each `.md` file becomes a document. We store the content and the relative filepath (this becomes our source citation later).

We use `glob` with `recursive=True` to catch files in subdirectories too.

In [2]:
def load_markdown_files(directory: str) -> list[dict]:
    """Load all .md files from a directory tree."""
    documents = []
    md_files = glob.glob(os.path.join(directory, "**/*.md"), recursive=True)

    for filepath in sorted(md_files):
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read().strip()
        if content:
            rel_path = os.path.relpath(filepath, directory)
            documents.append({"content": content, "source": rel_path})

    return documents


documents = load_markdown_files(DATASET_DIR)
print(f"Loaded {len(documents)} files")
print(f"\nFirst 5 files:")
for doc in documents[:5]:
    print(f"  {doc['source']} ({len(doc['content'])} chars)")

Loaded 50 files

First 5 files:
  about_direction.md (4681 chars)
  about_editing-handbook_edit-team-page.md (17284 chars)
  about_editing-handbook_practical-handbook-edits.md (13164 chars)
  about_escalation.md (5441 chars)
  about_handbook-usage.md (34913 chars)


## 1.2 Chunking Strategy

This is the most consequential design decision in the whole pipeline.

**Why chunk at all?**  
The embedding model has a token limit (~256 tokens for MiniLM). Entire files are too long to embed meaningfully. We need smaller pieces that each capture a coherent idea.

**Our approach: Paragraph-aware chunking with overlap**
- Split on double newlines (`\n\n`) to respect paragraph boundaries
- Accumulate paragraphs until we hit ~1000 characters
- When we start a new chunk, carry over the last ~200 characters from the previous chunk

**Why not just split every N characters?**  
Naive splitting cuts mid-sentence. A chunk that starts with "...the escalation channel" and ends with "You should also consi..." is useless for retrieval. Paragraph boundaries are natural semantic boundaries.

**Why overlap?**  
If the answer to a question spans two chunks, the overlap ensures both chunks contain enough context to be retrievable.

**Tunable parameters:**
- `CHUNK_SIZE`: Target chunk size in characters. 1000 is a good starting point.
- `CHUNK_OVERLAP`: How many characters to carry over. 200 gives ~20% overlap.

In [3]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


def chunk_document(text: str, source: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> list[dict]:
    """
    Split a document into overlapping chunks, breaking on paragraph boundaries.
    
    Returns a list of dicts with 'text', 'source', and 'chunk_id' keys.
    """
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""
    chunk_index = 0

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # If adding this paragraph would exceed chunk_size, save current and start new
        if len(current_chunk) + len(para) + 2 > chunk_size and current_chunk:
            chunks.append({
                "text": current_chunk.strip(),
                "source": source,
                "chunk_id": f"{source}::chunk_{chunk_index}"
            })
            chunk_index += 1

            # Overlap: carry the tail of the previous chunk into the new one
            if overlap > 0 and len(current_chunk) > overlap:
                current_chunk = current_chunk[-overlap:] + "\n\n" + para
            else:
                current_chunk = para
        else:
            current_chunk = current_chunk + "\n\n" + para if current_chunk else para

    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append({
            "text": current_chunk.strip(),
            "source": source,
            "chunk_id": f"{source}::chunk_{chunk_index}"
        })

    return chunks

In [4]:
# Chunk all documents
all_chunks = []
for doc in documents:
    doc_chunks = chunk_document(doc["content"], doc["source"])
    all_chunks.extend(doc_chunks)

# Stats
chunk_lengths = [len(c["text"]) for c in all_chunks]
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk size: {sum(chunk_lengths) // len(chunk_lengths)} chars")
print(f"Min: {min(chunk_lengths)}, Max: {max(chunk_lengths)}")

Total chunks: 802
Avg chunk size: 995 chars
Min: 154, Max: 21726


In [ ]:
# Inspect a few chunks to verify quality
print("=" * 60)
print("SAMPLE CHUNK 0:")
print("=" * 60)
print(f"Source: {all_chunks[0]['source']}")
print(f"Length: {len(all_chunks[0]['text'])} chars")
print(all_chunks[0]["text"][:500])
print("\n")
print("=" * 60)
print("SAMPLE CHUNK 10:")
print("=" * 60)
print(f"Source: {all_chunks[10]['source']}")
print(f"Length: {len(all_chunks[10]['text'])} chars")
print(all_chunks[10]["text"][:500])

### Checkpoint

Before moving to embedding, verify:
- Are chunks roughly 500-1200 chars? (Some will be shorter if a section is small, that's fine)
- Do chunks start and end at sensible boundaries? (Not mid-sentence)
- Does each chunk have the correct source file attributed?

If chunks look wrong, adjust `CHUNK_SIZE` and `CHUNK_OVERLAP` above and re-run.

## 1.3 Embed and Store in ChromaDB

**What's happening under the hood:**
- ChromaDB's default embedding function uses `all-MiniLM-L6-v2`, a small but effective sentence transformer
- Each chunk gets converted into a 384-dimensional vector
- These vectors are stored alongside the original text and metadata (source file)
- ChromaDB uses HNSW indexing for fast approximate nearest-neighbor search

**We use cosine similarity** (via `hnsw:space = cosine`), which measures the angle between vectors. Two chunks about "remote work" will have vectors pointing in a similar direction, regardless of the exact words used.

**PersistentClient** saves the database to disk so we don't need to re-embed every time. This folder is what you'll upload to Hugging Face Spaces.

In [5]:
!pip install chromadb

  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 50.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 39.3 MB/s  0:00:00m0:00:010:01
Using cached opentelemetry_api-1.40.0-py3-none-any.whl (68 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.8 MB/s  0:00:00
Using cached uvicorn-0.42.0-py3-none-any.whl (68 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29/29 [chromadb]/29 [chromadb]ce-hub]


In [6]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "gitlab_handbook"

# Default embedding function = all-MiniLM-L6-v2
ef = embedding_functions.DefaultEmbeddingFunction()

# Persistent client saves to disk
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete and recreate to start fresh
try:
    client.delete_collection(COLLECTION_NAME)
    print("Deleted existing collection.")
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}  # cosine similarity
)

print(f"Collection '{COLLECTION_NAME}' ready.")

Collection 'gitlab_handbook' ready.


In [7]:
# Add chunks in batches (ChromaDB recommends batches of ~500)
batch_size = 500

for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i : i + batch_size]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{"source": c["source"]} for c in batch],
    )
    print(f"Batch {i // batch_size + 1}: added {len(batch)} chunks")

print(f"\nDone. Total chunks in collection: {collection.count()}")

Batch 1: added 500 chunks
Batch 2: added 302 chunks

Done. Total chunks in collection: 802


## 1.4 Sanity Check: Query the Vector DB

Before moving to Step 2, let's verify the database works. We'll run a few test queries and inspect:
- Which chunks come back
- What distance scores they have (lower = more similar for cosine)
- Whether the source files make sense

This is the most important cell in this notebook. If retrieval is bad here, nothing downstream will fix it.

In [8]:
def test_query(query: str, n_results: int = 5):
    """Run a query against ChromaDB and display results."""
    results = collection.query(query_texts=[query], n_results=n_results)
    
    print(f"Query: '{query}'")
    print("-" * 60)
    
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]
        
        print(f"\n[{i+1}] Distance: {distance:.4f} | Source: {source}")
        print(f"    {doc[:150]}...")
    
    print("\n" + "=" * 60 + "\n")

In [9]:
# Test with questions that SHOULD find relevant content
test_query("What does handbook first mean?")
test_query("When should I escalate a handbook issue?")
test_query("How do I add my photo to the team page?")

Query: 'What does handbook first mean?'
------------------------------------------------------------

[1] Distance: 0.3118 | Source: about_handbook-usage.md
    s to be handbook first, please refer to the [why handbook first](/handbook/about/handbook-usage/#why-handbook-first) section of this page.

**Skills a...

[2] Distance: 0.3576 | Source: about_handbook-usage.md
    y add to or refactor the handbook to have a proper foundation. But, it saves time in the long run, and this communication is essential to our ability ...

[3] Distance: 0.4210 | Source: about_handbook-usage.md
    cations for other potentially affected processes.

Having a **"handbook first"** mentality ensures there is no duplication; the handbook is always up ...

[4] Distance: 0.4781 | Source: about_handbook-usage.md
    ecifically to mirror handbook information). This approach shows a [bias towards asynchronous communication](/handbook/values/#bias-towards-asynchronou...

[5] Distance: 0.4905 | Source: about_handb

In [10]:
# Test with the hybrid work question that failed before
test_query("What is GitLab's hybrid work policy?")
test_query("Does GitLab allow hybrid work?")
test_query("What is GitLab's remote work policy?")

Query: 'What is GitLab's hybrid work policy?'
------------------------------------------------------------

[1] Distance: 0.3845 | Source: company_culture_all-remote_getting-started.md
    operate differently.

So differently, in fact, that many of GitLab's most effective processes would be discouraged or forbidden in conventional corpor...

[2] Distance: 0.3961 | Source: company_culture_all-remote_asynchronous.md
    here businesses are increasingly remote. GitLab's seeks to more clearly define and operationalize asynchronous communication.

Over time, GitLab, the ...

[3] Distance: 0.4138 | Source: business-technology_entapps-documentation_policies_gitlab-business-continuity-plan.md
    ---
title: "Business Continuity Plan"
controlled_document: true
tags:
  - security_policy
  - security_policy_cpir
---

{{< label name="Visibility: Au...

[4] Distance: 0.4142 | Source: company_culture_all-remote_getting-started.md
    c together. Several GitLab team members came together to create [A

In [11]:
# Test with questions that SHOULD NOT find relevant content (negative examples)
test_query("What is the capital of France?")
test_query("What is GitLab's stock price?")

Query: 'What is the capital of France?'
------------------------------------------------------------

[1] Distance: 0.8954 | Source: communication_confidentiality-levels.md
    a code name for the project. For consistency and to make it easier to identify the genesis of these projects and their organizational affiliations, we...

[2] Distance: 0.9024 | Source: communication_chat.md
    d with a `g_`) correspond to a [DevOps Stage group](/handbook/product/categories/#hierarchy) and other [engineering departments and teams](/handbook/e...

[3] Distance: 0.9061 | Source: about_handbook-usage.md
    ontent.
- Note: A weakness of [FAQs](/handbook/communication/#structure-content-instead-of-using-faqs) is that questions are often asked in biased or ...

[4] Distance: 0.9128 | Source: business-technology_engineering_infrastructure_reference-links.md
    ets mapped to the milestone for the time period that most of the work takes place. This is used as our lightweight version of time tracking a

## What to Look For

**Positive queries:**
- Top results should have low distances (under ~0.5 for cosine)
- Source files should match what you'd expect
- The chunk text should actually contain the answer

**The hybrid work query:**
- Look at the distances. If the remote work chunks come back at 0.3-0.5, that's retrievable.
- If they come back at 0.8+, the embedding model isn't connecting "hybrid" to "remote"
- This tells us where to set our relevance threshold in Step 2

**Negative queries:**
- These will still return 5 results (ChromaDB always returns something)
- But the distances should be noticeably higher (0.8+)
- The gap between positive and negative distances is what our threshold needs to exploit

---

**Next step:** Once you're happy with retrieval quality, move to `step2_retrieval.ipynb` where we build the retrieval function with threshold tuning.